# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library. We'll inspect metadata and record sets, process records, and visualize the clinical/genomic data in tabular form.

### Dataset Source
The dataset source is provided via the following Croissant schema URL:

https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Published: {metadata.datePublished}\n")
print(f"Version: {metadata.version}\n")
print(f"Authors (@id): {[author['@id'] for author in metadata.author]}\n")

## 2. Data Overview
Explore available record sets, fields, and their `@id`s.

Let's enumerate all available record sets. Since the schema is FAIR^2, each entity has a unique `@id`. We'll print the `@id`s of each record set and list their fields (columns, also referenced by `@id`).

In [ ]:
# Enumerate record sets and their structure from metadata
# mlcroissant exposes record_sets under `dataset.metadata.recordSet`

record_sets = getattr(metadata, 'recordSet', [])

# If 'recordSet' is empty, try to fetch by loading via dataset.records (which loads via Croissant schema definition)
if not record_sets:
    print("No recordSet found in metadata. We'll try to infer record sets from data docs.")
    record_sets = [rs['@id'] for rs in dataset.record_sets]
else:
    # record_sets might actually be a list of dictionaries (with @id and other keys)
    record_sets = [rs['@id'] if isinstance(rs, dict) else rs for rs in record_sets]

print("Record Set @ids:")
for rs_id in record_sets:
    print(f"- {rs_id}")
    fields = dataset.fields(record_set=rs_id)
    print(f"  Fields (columns) @ids:")
    for field in fields:
        print(f"    - {field['@id']} (name: {field.get('name', '')})")

## 3. Data Extraction
Load tabular data from a specific record set into a DataFrame for analysis.

For clinical data, hormone data, or genetic variant tables, select the appropriate record set and column `@id`s. We'll demonstrate loading all available record sets.

In [ ]:
# Load data from each record set into pandas DataFrames
dataframes = {}

for rs_id in record_sets:
    print(f"Loading record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"  Columns (@id): {df.columns.tolist()}")
    print(df.head(), "\n")

# For demonstration, select the first record set
selected_rs = record_sets[0] if record_sets else None
if selected_rs:
    print(f"Sample columns for record set {selected_rs}: {dataframes[selected_rs].columns.tolist()}")
    dataframes[selected_rs].head()

## 4. Exploratory Data Analysis (EDA)
Process and filter records: select a numeric field, filter records, normalize values, and group by categorical attributes (all by `@id`).

We'll show filtering on a numeric column (`age` or hormone value), normalizing it, and grouping (if a grouping field exists such as "phenotype" or "diagnosis").

In [ ]:
# Choose the numeric field @id (e.g. age at diagnosis, hormone level, etc.)
# Choose a group field @id (e.g. inferred phenotype, if present)

numeric_field_id = None
group_field_id = None

df = dataframes[selected_rs] if selected_rs else None
print(f"Columns in DataFrame: {df.columns.tolist()}\n")

# Try to infer numeric and group fields from columns
for col in df.columns:
    # Heuristically pick 'age', hormone value, or numeric field
    if 'age' in col.lower() or 'level' in col.lower() or 'value' in col.lower():
        numeric_field_id = col
    if 'phenotype' in col.lower() or 'diagnosis' in col.lower():
        group_field_id = col

# If unavailable, manually set from column list
if not numeric_field_id:
    numeric_field_id = df.columns[0]  # Take first column
if not group_field_id and len(df.columns) > 1:
    group_field_id = df.columns[1]   # Take second column

print(f"Using numeric field @id: {numeric_field_id}")
print(f"Grouping field @id: {group_field_id}\n")

# Filtering based on numeric value
threshold = 10
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"Column {numeric_field_id} is not numeric; EDA skipped.")

## 5. Visualization
Visualize distributions or relationships between selected fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of numeric field
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    plt.figure(figsize=(6, 4))
    sns.histplot(df[numeric_field_id], bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

# If group field present
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(6, 4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
In this notebook, we loaded the clinical/genomic FAIR^2 dataset using `mlcroissant`, explored tabular data via record set `@id`s, filtered and normalized numeric values, and visualized distributions and relationships. Each entity was referenced by its unique `@id` for reproducibility and traceability.

This structured approach facilitates transparent and FAIR-compliant data analysis, supporting rare disease studies and clinical research. Further exploration can be performed by linking more columns and expanding cohort (if available).